# Day 21 - Selection and Top-K

Yesterday we sorted. Today we ask a smaller question: **what are the k largest
elements?** - and we refuse to sort to answer it.

The gap is not academic. Every token an LLM generates runs a top-k over its
vocabulary, and every attention head with sparse attention runs a top-k over
its scores. That is a few thousand top-k calls per second, per GPU, so the
kernels that do it have been optimised into a shape that looks nothing like
the textbook algorithm - and that shape is what the second half of this
notebook is about.

We go: heap and quickselect (the classic toolkit), then the order-preserving
float key, then radix select, then the two ways real GPU kernels parallelise
it - one block per row, and many blocks sharing one row - including the
grid-wide barrier they need and the deadlock that barrier caused in
production.

In [1]:
import heapq
import random
import struct
import threading
import time

## 1. Sorting is the wrong tool

To get the top 8 of 200,000 numbers, sorting does millions of comparisons and
then throws 199,992 answers away. Three algorithms do better, each with a
different reason to exist.

In [2]:
def sort_topk(a, k):
    """The obvious one.  O(n log n) to answer an O(n) question."""
    return sorted(a, reverse=True)[:k]


def heap_topk(a, k, stats=None):
    """Keep a min-heap of size k.  The root is the *weakest* survivor, so one
    comparison per new element decides whether it is worth keeping.

    O(n log k) time and - the part that matters - O(k) memory.  This is the
    only algorithm here that never needs the whole input at once, so it is
    what you reach for when `a` arrives over a network."""
    h = []
    for x in a:
        if stats is not None:
            stats['cmp'] += 1
        if len(h) < k:
            heapq.heappush(h, x)
        elif x > h[0]:                   # beats the weakest survivor
            heapq.heapreplace(h, x)
        # else: discarded without ever entering the heap
    return sorted(h, reverse=True)


def heap_sort(a, stats=None):
    """Heap sort, for contrast: build a max-heap in place, then repeatedly
    swap the root to the end and sift down.  O(n log n) worst case with O(1)
    extra memory - the only sort we have seen with both.

    Written on a max-heap so it sorts ascending in place."""
    a = list(a)
    n = len(a)

    def sift(root, end):
        while True:
            child = 2 * root + 1
            if child >= end:
                return
            if child + 1 < end:
                if stats is not None:
                    stats['cmp'] += 1
                if a[child + 1] > a[child]:
                    child += 1           # always compare against the LARGER child
            if stats is not None:
                stats['cmp'] += 1
            if a[root] >= a[child]:
                return
            a[root], a[child] = a[child], a[root]
            root = child

    for i in range(n // 2 - 1, -1, -1):  # build: O(n), not O(n log n)
        sift(i, n)
    for end in range(n - 1, 0, -1):
        a[0], a[end] = a[end], a[0]      # largest goes to its final slot
        sift(0, end)
    return a


def quickselect(a, k, rng=None, stats=None):
    """The k largest, by partitioning and recursing into ONE side.

    Quick sort recurses into both halves and pays O(n log n).  Selection only
    ever needs one of them, and n + n/2 + n/4 + ... = 2n, so the average cost
    collapses to O(n).  That halving is the entire idea.

    Three-way partition, exactly as on day 20: without it a row of equal
    scores - which is what a masked attention row looks like - degrades to
    O(n^2)."""
    rng = rng or random
    a = list(a)
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        p = a[rng.randint(lo, hi)]       # random pivot: no input is special
        lt, i, gt = lo, lo, hi
        while i <= gt:                   # Dutch national flag, descending
            if stats is not None:
                stats['cmp'] += 1
            if a[i] > p:
                a[lt], a[i] = a[i], a[lt]; lt += 1; i += 1
            elif a[i] < p:
                a[i], a[gt] = a[gt], a[i]; gt -= 1     # do NOT advance i
            else:
                i += 1
        # a[lo:lt] > p,  a[lt:gt+1] == p,  a[gt+1:hi+1] < p
        if k <= lt - lo:
            hi = lt - 1                  # answer lives entirely in the > side
        elif k <= gt + 1 - lo:
            break                        # the boundary falls inside the ties
        else:
            k -= gt + 1 - lo             # discard > and ==, keep hunting
            lo = gt + 1
    return a


def median_of_medians(a, k, stats=None):
    """Quickselect with a *guaranteed* good pivot: O(n) worst case.

    Split into groups of 5, take each group's median, then recursively take
    the median of those.  That pivot is provably better than 30% and worse
    than 30% of the input, so the recursion always throws away at least three
    tenths - no adversary can build a bad input.

    It is a genuine theoretical result and almost nobody ships it: the
    constant factor is several times a random pivot, and a random pivot is
    already unbeatable in practice.  Worth knowing exactly because it shows
    the difference between 'no bad input exists' and 'bad inputs are too
    unlikely to care about'."""
    a = list(a)

    def select(items, j):                # j-th smallest, 0-indexed
        if len(items) <= 5:
            return sorted(items)[j]
        medians = []
        for i in range(0, len(items), 5):
            g = sorted(items[i:i + 5])
            medians.append(g[len(g) // 2])
        pivot = select(medians, len(medians) // 2)
        lo = [x for x in items if x < pivot]
        eq = [x for x in items if x == pivot]
        hi = [x for x in items if x > pivot]
        if stats is not None:
            stats['cmp'] += len(items)
        if j < len(lo):
            return select(lo, j)
        if j < len(lo) + len(eq):
            return pivot
        return select(hi, j - len(lo) - len(eq))

    kth = select(a, len(a) - k)          # k-th largest = (n-k)-th smallest
    return kth


def reservoir_sample(stream, k, rng=None):
    """A uniform sample of k items from a stream of unknown length, in O(k).

    Keep the first k.  For the i-th item (0-indexed), keep it with probability
    k/(i+1), evicting a uniformly random survivor.  The invariant is that
    after seeing n items every item is present with probability exactly k/n -
    which is what 'uniform' means, and it holds without ever knowing n.

    This is the selection problem's sibling: same 'one pass, bounded memory'
    shape as heap_topk, except the score is random instead of given."""
    rng = rng or random
    res = []
    for i, x in enumerate(stream):
        if i < k:
            res.append(x)
        else:
            j = rng.randint(0, i)        # inclusive
            if j < k:
                res[j] = x
    return res

In [3]:
rng = random.Random(20260821)
n, k = 200_000, 8
data = [rng.random() for _ in range(n)]
truth = sorted(data, reverse=True)[:k]

st_sort, st_heap, st_qs, st_mom = ({'cmp': 0} for _ in range(4))
heap_sort(list(data), st_sort)
got_heap = heap_topk(data, k, st_heap)
got_qs = quickselect(data, k, random.Random(7), st_qs)[:k]
got_mom = median_of_medians(data, k, st_mom)

print(f'n = {n:,}, k = {k}')
print(f'  heap sort everything   {st_sort["cmp"]:>10,} comparisons')
print(f'  size-k min-heap        {st_heap["cmp"]:>10,} comparisons, O(k) memory, one pass')
print(f'  quickselect            {st_qs["cmp"]:>10,} comparisons')
print(f'  median-of-medians      {st_mom["cmp"]:>10,} comparisons (O(n) worst case)')
assert got_heap == truth and sorted(got_qs, reverse=True) == truth and got_mom == truth[-1]
print('all four agree on the top-8')

n = 200,000, k = 8
  heap sort everything    6,439,409 comparisons
  size-k min-heap           200,000 comparisons, O(k) memory, one pass
  quickselect               244,798 comparisons
  median-of-medians         653,431 comparisons (O(n) worst case)
all four agree on the top-8


Quickselect is the one to internalise: quick sort recurses into **both** halves
and pays `O(n log n)`; selection only ever needs one of them, and
`n + n/2 + n/4 + ... = 2n`. That halving is the whole idea.

The heap is the one you actually reach for when the data arrives over a
network, because it is the only variant here that never needs the whole input
in memory at once.

## 2. The order-preserving key

Everything below stops comparing floats and starts looking at their bits. That
only works if the bit pattern sorts the same way the number does, which for
IEEE-754 it *almost* does: positive floats already compare correctly as
unsigned integers, and negative floats compare exactly backwards.

In [4]:
def to_ordered(x):
    """float32 -> uint32 such that a < b  <=>  to_ordered(a) < to_ordered(b)."""
    bits = struct.unpack('<I', struct.pack('<f', x))[0]
    if bits >> 31:                       # negative: reverse the whole range
        return (~bits) & 0xFFFFFFFF
    return bits | 0x80000000             # positive: lift above every negative


def from_ordered(u):
    """The inverse of to_ordered - the map loses nothing."""
    bits = (u ^ 0x80000000) if (u >> 31) else ((~u) & 0xFFFFFFFF)
    return struct.unpack('<f', struct.pack('<I', bits))[0]


def f32(x):
    """Round a Python float to float32, so the bit tricks are exact."""
    return struct.unpack('<f', struct.pack('<f', x))[0]

In [5]:
probe = [-float('inf'), -3.5, -1.0, -0.0, 0.0, 1.0, 3.5, float('inf')]
print('     value       float32 bits   ordered key')
for v in probe:
    b = struct.unpack('<I', struct.pack('<f', f32(v)))[0]
    print(f'  {v:>10}     0x{b:08x}    0x{to_ordered(v):08x}')

keys = [to_ordered(v) for v in probe]
assert keys == sorted(keys)
assert all(from_ordered(to_ordered(v)) == f32(v) for v in probe if v == v)
print('\nmonotone, and invertible - nothing is lost')

     value       float32 bits   ordered key
        -inf     0xff800000    0x007fffff
        -3.5     0xc0600000    0x3f9fffff
        -1.0     0xbf800000    0x407fffff
        -0.0     0x80000000    0x7fffffff
         0.0     0x00000000    0x80000000
         1.0     0x3f800000    0xbf800000
         3.5     0x40600000    0xc0600000
         inf     0x7f800000    0xff800000

monotone, and invertible - nothing is lost


Two details worth keeping. The map is **bijective**, so a radix pass over the
keys is not an approximation of a comparison - it *is* one. And `-0.0` and
`+0.0` get different keys, even though Python says they are equal; every radix
select shares that wrinkle.

## 3. Radix select

Radix select never compares two elements. It bins them by a slice of their
bits, counts the bins, and uses the counts alone to work out which bin the
k-th element lives in. Then it recurses into that one bin with the next slice.

The consequence that makes it the GPU algorithm of choice: **a round moves 256
counters, not n elements.** Quickselect has to physically swap elements to
partition them. Radix select never moves the data at all.

In [6]:
RADIX_BITS = 8
RADIX = 1 << RADIX_BITS


def radix_select(values, k, bits=RADIX_BITS, trace=None):
    """Find the pivot - the ordered key of the k-th largest - MSB first.

    Returns (pivot, above, equal) where `above` is the list of indices
    strictly greater than the pivot and `equal` is the list tied with it.
    The answer is `above` plus (k - len(above)) of `equal`; which ones, see
    section 4.
    """
    ordered = [to_ordered(v) for v in values]
    rounds = (32 + bits - 1) // bits
    radix = 1 << bits
    prefix = 0                            # the high bits decided so far
    remaining = k                         # how many of the top-k live in here
    for r in range(rounds):
        shift = 32 - (r + 1) * bits
        mask = (0xFFFFFFFF << (shift + bits)) & 0xFFFFFFFF   # the decided bits
        hist = [0] * radix
        for u in ordered:
            if (u & mask) == prefix:      # only elements still in the running
                hist[(u >> shift) & (radix - 1)] += 1
        # suffix sum: ge[b] = how many candidates have bucket >= b
        ge = [0] * (radix + 1)
        for b in range(radix - 1, -1, -1):
            ge[b] = ge[b + 1] + hist[b]
        # exactly one bucket straddles the boundary
        bucket = next(b for b in range(radix)
                      if ge[b] >= remaining and ge[b + 1] < remaining)
        if trace is not None:
            trace.append({'round': r, 'shift': shift, 'hist': hist,
                          'bucket': bucket, 'candidates': ge[0],
                          'gt': ge[bucket + 1], 'eq': hist[bucket],
                          'remaining': remaining, 'prefix': prefix})
        prefix |= bucket << shift
        remaining -= ge[bucket + 1]       # those are already guaranteed winners
    pivot = prefix
    above = [i for i, u in enumerate(ordered) if u > pivot]
    equal = [i for i, u in enumerate(ordered) if u == pivot]
    return pivot, above, equal


def radix_topk(values, k, bits=RADIX_BITS, trace=None):
    """The k largest indices, via radix select.  Ties resolved smallest-index."""
    if k >= len(values):
        return list(range(len(values)))
    pivot, above, equal = radix_select(values, k, bits, trace)
    return above + equal[:k - len(above)]

In [7]:
r2 = random.Random(11)
row = [f32(r2.gauss(0, 2)) for _ in range(4000)]

trace = []
pivot, above, equal = radix_select(row, 32, trace=trace)
print('4000 attention-like scores, k = 32')
print('  round  shift  bucket   candidates   still to find')
for t in trace:
    print(f'    {t["round"]}     {t["shift"]:>2}     {t["bucket"]:>3}   '
          f'{t["candidates"]:>10,}   {t["remaining"]:>13}')
print(f'\npivot = {from_ordered(pivot):.6f}, {len(above)} above it, {len(equal)} tied')

sel = radix_topk(row, 32)
assert sorted((row[i] for i in sel), reverse=True) == sorted(row, reverse=True)[:32]
print('matches a full sort')

4000 attention-like scores, k = 32
  round  shift  bucket   candidates   still to find
    0     24     192        4,000              32
    1     16     155          653              32
    2      8     155            2               2
    3      0     231            1               1

pivot = 4.862781, 31 above it, 1 tied
matches a full sort


Watch which column shrinks: the candidate count, from 4000 to a handful, in
four rounds. Watch which one does not exist: any column counting moved data.

## 4. Parallel top-k, version A: one block owns one row

Now the real kernel. A GPU block gets a whole row and wants to finish it
without ever going back to global memory. The move is a **filter pass**: one
streaming pass over the row that throws away everything provably out, keeping
only a few hundred candidates that fit in shared memory.

To bin cheaply the kernel narrows each float to 16 bits first and histograms
the top byte of *that*. Narrowing is monotone non-decreasing, so it can merge
values but never reorder them - and merging is exactly what the later
refinement rounds undo.

In [8]:
def to_fp16_bits(x):
    """float32 -> the 16 raw bits of float16, round-to-nearest-even."""
    b = struct.unpack('<I', struct.pack('<f', f32(x)))[0]
    sign = (b >> 16) & 0x8000
    exp = (b >> 23) & 0xFF
    man = b & 0x7FFFFF
    if exp == 0xFF:                                  # inf / nan
        return sign | 0x7C00 | (0x200 if man else 0)
    e = exp - 127 + 15                               # rebias 127 -> 15
    if e >= 0x1F:                                    # overflows fp16 -> inf
        return sign | 0x7C00
    if e <= 0:                                       # subnormal or zero
        if e < -10:
            return sign
        man |= 0x800000                              # restore implicit 1
        shift = 14 - e
        half = man >> shift
        rem = man & ((1 << shift) - 1)
        tie = 1 << (shift - 1)
        if rem > tie or (rem == tie and (half & 1)):
            half += 1
        return sign | half
    half = (e << 10) | (man >> 13)
    rem = man & 0x1FFF                               # the 13 dropped bits
    if rem > 0x1000 or (rem == 0x1000 and (half & 1)):
        half += 1                                    # may carry into exponent
    return sign | half


def ordered16(x):
    """The same sign trick, on the 16 bits of float16."""
    b = to_fp16_bits(x)
    return ((~b) & 0xFFFF) if (b >> 15) else (b | 0x8000)


def coarse_key(x, bits=8):
    """The first-round bucket id: the top `bits` of the ordered float16 key."""
    return ordered16(x) >> (16 - bits)


def fp16_bits_to_float(b):
    """The 16 bits of a float16 -> the float32 it represents."""
    sign = -1.0 if (b >> 15) else 1.0
    exp = (b >> 10) & 0x1F
    man = b & 0x3FF
    if exp == 0x1F:
        return sign * (float('inf') if man == 0 else float('nan'))
    if exp == 0:
        return sign * man * 2.0 ** -24                     # subnormal
    return sign * (1.0 + man / 1024.0) * 2.0 ** (exp - 15)


def unordered16(key):
    """Inverse of ordered16: the ordered uint16 back to raw float16 bits."""
    return (key & 0x7FFF) if (key >> 15) else ((~key) & 0xFFFF)


def coarse_bin_lower_bound(b, bits=8):
    """The smallest float32 v for which coarse_key(v, bits) >= b.

    So `v >= coarse_bin_lower_bound(t)` replaces `coarse_key(v) >= t`, and
    two of these bracket the threshold bin exactly."""
    if b <= 0:
        return float('-inf')
    if b >= (1 << bits):
        return float('inf')
    key = b << (16 - bits)                # smallest ordered16 key inside bin b
    hi = fp16_bits_to_float(unordered16(key))
    lo = fp16_bits_to_float(unordered16(key - 1))
    # The ends of the key space are inf and NaN, and a midpoint against those
    # is meaningless.  Getting this wrong is not academic: a NaN bound makes
    # BOTH of the collect pass's compares fail, the row comes back with fewer
    # than k entries, and whatever consumes those indices reads garbage.
    if hi != hi:                          # bin sits in a NaN region
        # NaN lives at BOTH ends of the key space - negative NaNs below every
        # finite value, positive NaNs above every finite value - so which
        # infinity to answer depends on which end we are at.
        return float('inf') if key >= 0x8000 else float('-inf')
    if hi == float('-inf'):
        return float('-inf')              # every finite value clears it
    if hi == float('inf'):
        # No midpoint available.  The boundary is half an ulp above the
        # largest finite float16, because that is where rounding tips to inf.
        prev = fp16_bits_to_float(unordered16(key - 2))
        return f32(lo + 0.5 * (lo - prev))
    if lo != lo or lo == float('-inf'):   # nothing finite below this bin
        return hi
    return f32(0.5 * (lo + hi))           # the round-to-nearest boundary

In [9]:
DTYPES = {
    # name        exact_bits  num_rounds  first_shift  coarse_is_prefix
    'float32':   (32,         4,          24,          False),
    'float16':   (16,         1,          0,           True),
    'bfloat16':  (16,         1,          0,           True),
}

# For fp16 and bf16 the coarse key IS the top byte of the exact key, so the
# coarse pass already decided 8 of the 16 bits and one round of 8 finishes it.
#
# For fp32 the coarse key lives in a DIFFERENT key space - the value was
# narrowed to fp16 first - so it tells you which *bin* you are in but not a
# single bit of the fp32 key.  Refinement has to resolve all 32 bits from the
# top: shifts 24, 16, 8, 0.  Four rounds.  You buy a well-spread histogram
# with the coarse pass and you pay for it with rounds you cannot skip.

FILTER_CAPACITY = 16384       # candidate slots in shared memory (flashinfer's)


def exact_key(v, dtype='float32'):
    """The lossless sortable key for the refinement rounds."""
    if dtype == 'float32':
        return to_ordered(v)
    if dtype == 'float16':
        return ordered16(v)
    if dtype == 'bfloat16':                     # bf16 = the top 16 bits of fp32
        u = to_ordered(v)
        return u >> 16
    raise ValueError(dtype)


def filtered_topk(values, k, dtype='float32', capacity=FILTER_CAPACITY,
                  coarse_bits=8, stats=None):
    """Top-k for one row, the way a single GPU block does it.

    Returns the selected indices.  `stats` collects the numbers that make the
    argument: how many elements survived the filter, and how many rounds ran.
    """
    n = len(values)
    if k >= n:
        return list(range(n))
    exact_bits, num_rounds, first_shift, coarse_is_prefix = DTYPES[dtype]
    nbins = 1 << coarse_bits

    # How the coarse bin is derived is exactly what splits the two cases.
    # 16-bit inputs: the bin is literally the top byte of the exact key, so
    # the coarse pass has already decided half the bits.  fp32: the bin comes
    # from a narrowed copy and decides nothing about the fp32 key.
    if coarse_is_prefix:
        def bin_of(v):
            return exact_key(v, dtype) >> (16 - coarse_bits)
    else:
        def bin_of(v):
            return coarse_key(v, coarse_bits)

    # -- pass 1: the coarse histogram.  One streaming read of the row. --------
    hist = [0] * nbins
    for v in values:
        hist[bin_of(v)] += 1
    ge = [0] * (nbins + 1)
    for b in range(nbins - 1, -1, -1):
        ge[b] = ge[b + 1] + hist[b]
    threshold_bin = next(b for b in range(nbins)
                         if ge[b] >= k and ge[b + 1] < k)
    remaining = k - ge[threshold_bin + 1]       # still owed from inside the bin

    # -- the boundary trick: two floats replace a per-element key computation -
    #
    # Pass 2 is compute-bound, so recomputing fp32->fp16->bits for every
    # element just to recover its bin is real money.  Instead project the
    # threshold bin's two edges back into fp32 ONCE, and the whole
    # classification becomes two compares against loop-invariant registers.
    if coarse_is_prefix:
        v_hi = ((threshold_bin + 1) << (16 - coarse_bits))
        v_lo = (threshold_bin << (16 - coarse_bits))

        def rank(v):
            return exact_key(v, dtype)
    else:
        v_hi = coarse_bin_lower_bound(threshold_bin + 1, coarse_bits)
        v_lo = coarse_bin_lower_bound(threshold_bin, coarse_bits)

        def rank(v):
            return v                            # a bare float compare

    # -- pass 2: filter.  Three-way, and the only pass that touches the row
    #    again.  Everything above the threshold bin is already a winner; we
    #    never have to look at it or at anything below the bin ever again. ----
    winners, candidates = [], []
    for i, v in enumerate(values):
        rv = rank(v)
        if rv >= v_hi:
            winners.append(i)                   # guaranteed, no further work
        elif rv >= v_lo:
            candidates.append(i)                # needs the exact key to rank
    overflow = len(candidates) > capacity

    if stats is not None:
        stats.update(n=n, k=k, dtype=dtype, threshold_bin=threshold_bin,
                     v_lo=v_lo, v_hi=v_hi, winners=len(winners),
                     candidates=len(candidates), remaining=remaining,
                     overflow=overflow, coarse_bins_used=sum(1 for h in hist if h),
                     rounds=0)

    # A subtlety worth pausing on: `remaining` was computed from the fp16
    # histogram, but `winners` was computed from the fp32 boundary compare.
    # Rounding can move a boundary element between the two sets.  The kernels
    # trust the COLLECT counts, not the histogram counts - one classification
    # has to be the ground truth and it has to be the one that actually wrote
    # the output.
    remaining = k - len(winners)
    if remaining <= 0:
        return winners[:k]

    # -- refinement: radix select, but only over the candidate buffer --------
    keys = [(exact_key(values[i], dtype), i) for i in candidates]
    prefix, need = 0, remaining
    live = keys
    for r in range(num_rounds):
        shift = first_shift - 8 * r
        h = [0] * 256
        for u, _ in live:
            h[(u >> shift) & 0xFF] += 1
        g = [0] * 257
        for b in range(255, -1, -1):
            g[b] = g[b + 1] + h[b]
        bucket = next(b for b in range(256) if g[b] >= need and g[b + 1] < need)
        if stats is not None:
            stats['rounds'] = r + 1
        prefix |= bucket << shift
        need -= g[bucket + 1]
        nxt = [(u, i) for u, i in live if ((u >> shift) & 0xFF) == bucket]
        winners.extend(i for u, i in live if ((u >> shift) & 0xFF) > bucket)
        live = nxt
        if need == 0:
            break
    # whatever is left is tied with the pivot; take `need` of them
    winners.extend(i for _, i in live[:need])
    return winners

In [10]:
r3 = random.Random(5)
wide = [f32(r3.gauss(0, 2)) for _ in range(20_000)]

st = {}
sel = filtered_topk(wide, 64, 'float32', stats=st)
assert sorted((wide[i] for i in sel), reverse=True) == sorted(wide, reverse=True)[:64]

print(f'n = {len(wide):,}, k = 64')
print(f'  occupied coarse bins   {st["coarse_bins_used"]:>8}')
print(f'  threshold bin          {st["threshold_bin"]:>8}')
print(f'  v_hi (sure winners)    {st["v_hi"]:>8.4f}')
print(f'  v_lo (sure losers)     {st["v_lo"]:>8.4f}')
print(f'  guaranteed winners     {st["winners"]:>8,}')
print(f'  candidates kept        {st["candidates"]:>8,}  '
      f'({100.0 * st["candidates"] / len(wide):.2f}% of the row)')
print(f'  buffer overflowed      {str(st["overflow"]):>8}')

n = 20,000, k = 64
  occupied coarse bins        106
  threshold bin               197
  v_hi (sure winners)      5.9980
  v_lo (sure losers)       4.9980
  guaranteed winners           31
  candidates kept             103  (0.52% of the row)
  buffer overflowed         False


`v_lo` and `v_hi` are the trick worth stealing. The obvious way to classify an
element in the second pass is to recompute its coarse bin - a float32 to
float16 conversion plus bit twiddling, per element, in the innermost loop of a
bandwidth-bound kernel. Instead the kernel projects the threshold bin's two
edges back into float32 **once**, and the whole classification becomes two
float compares against loop-invariant registers.

Because the narrowing rounds to nearest, the true boundary is the *midpoint* of
two adjacent float16 values, not the bin edge itself.

### Why float32 needs four refinement rounds and bfloat16 needs one

This looks like a tuning constant and it is not. It falls out of one fact: for
a 16-bit dtype the coarse bin **is** the top byte of the exact key, so the
coarse pass has already decided 8 of the 16 bits. For float32 the coarse key
lives in a *different key space* - it came from a narrowed copy - so it decides
zero bits of the float32 key, and refinement has to resolve all 32 from the
top.

In [11]:
print('  dtype       exact bits  rounds  first shift  coarse key is a prefix?')
for name, (bits_, nr, fs, pref) in DTYPES.items():
    print(f'  {name:<10} {bits_:>10}  {nr:>6}  {fs:>11}  {"yes" if pref else "no":>12}')

print()
print('  the same 20,000 values, three dtypes:')
print('  dtype       coarse bins used   candidates kept   rounds')
for name in ('float32', 'float16', 'bfloat16'):
    s = {}
    got = filtered_topk(wide, 64, name, stats=s)
    assert len(got) == 64
    print(f'  {name:<10} {s["coarse_bins_used"]:>16}   {s["candidates"]:>15,}   {s["rounds"]:>6}')

  dtype       exact bits  rounds  first shift  coarse key is a prefix?
  float32            32       4           24            no
  float16            16       1            0           yes
  bfloat16           16       1            0           yes

  the same 20,000 values, three dtypes:
  dtype       coarse bins used   candidates kept   rounds
  float32                 106               103        4
  float16                 106               103        1
  bfloat16                 15             3,117        1


bfloat16 keeps far more candidates than float16 on the same row, and that is
the honest argument for narrowing in the first place. bfloat16 *is* the top 16
bits of float32, so its top byte is sign plus 7 exponent bits - and a row of
attention logits all shares an exponent, so they pile into a couple of dozen
bins. Narrowing to float16 spreads the same values across the full 256.

## 5. Parallel top-k, version B: many blocks share one row

Version A caps out at one block per row. Given four rows and a thousand cores,
most of the machine idles. Version B splits **one** row across several blocks,
which buys parallelism and costs the one thing a single block never needed: a
way for blocks to agree.

There is no `__syncthreads()` across blocks, so the kernel builds a grid-wide
barrier out of a single counter. Three details in it are load-bearing, and all
three are the kind of thing you only get right after getting it wrong.

In [12]:
class CountingBarrier:
    """The barrier a cooperative kernel actually uses, warts included.

    There is no `__syncthreads()` across blocks, so it is built by hand out of
    one shared counter.  Two details are load-bearing:

    * The counter is NEVER reset between rounds.  Each CTA keeps a private
      phase number and waits for `counter >= (phase+1) * num_ctas`.  A
      sense-reversing barrier would need a second variable and a second race.

    * The wait is `>=`, not `==`.  A CTA that is descheduled for a moment can
      wake up to find the counter has already sailed past its target; with
      `==` it would wait forever for an edge that already happened.
    """

    def __init__(self, num_ctas):
        self.num_ctas = num_ctas
        self.counter = 0
        self.lock = threading.Lock()
        self.max_spins = 0

    def arrive_and_wait(self, phase, deadline, spin_hook=None):
        with self.lock:
            self.counter += 1
        target = (phase + 1) * self.num_ctas
        spins = 0
        while True:
            if spin_hook is not None:
                spin_hook(spins)
            with self.lock:
                seen = self.counter
            if seen >= target:                  # ">=", not "=="
                self.max_spins = max(self.max_spins, spins)
                return True
            spins += 1
            if time.monotonic() > deadline:
                return False                    # caller decides: this is a hang
            # A real GPU spins on a cache line and costs nothing.  Python
            # threads share one interpreter lock, so an unyielding spin would
            # starve the very CTAs we are waiting for.  This sleep is an
            # artefact of the simulation, not of the algorithm.
            time.sleep(0.0002)

In [13]:
def multi_cta_topk(values, k, num_ctas=4, bits=8, reset_policy='last',
                   slow_cta=None, timeout=10.0, log=None):
    """Top-k for one row, split across `num_ctas` cooperating blocks.

    `reset_policy` picks who zeroes the shared counter so the next launch
    starts clean:

        'last'  - only the CTA that provably arrived last.  Correct.
        'first' - whoever finishes the final barrier first.  Deadlocks.

    Returns (indices, info).  If the barrier hangs, indices is None.
    """
    n = len(values)
    ordered = [to_ordered(v) for v in values]
    radix = 1 << bits
    rounds = (32 + bits - 1) // bits
    chunk = (n + num_ctas - 1) // num_ctas

    barrier = CountingBarrier(num_ctas)
    # the shared state that lives in global memory, one copy for the whole row
    # Three histogram buffers, not one and not two.  See the loop below.
    shared = {'hist': [[0] * radix for _ in range(3)],
              'out': [], 'out_lock': threading.Lock(),
              'hist_lock': threading.Lock(), 'deadlocked': False,
              'exit_arrivals': 0, 'reset_by': None}
    deadline = time.monotonic() + timeout
    per_cta = [None] * num_ctas

    def cta(c):
        lo, hi = c * chunk, min((c + 1) * chunk, n)
        mine = list(range(lo, hi))
        prefix, remaining, phase = 0, k, 0
        for r in range(rounds):
            shift = 32 - (r + 1) * bits
            mask = (0xFFFFFFFF << (shift + bits)) & 0xFFFFFFFF
            cur, nxt = r % 3, (r + 1) % 3
            local = [0] * radix
            for i in mine:
                if (ordered[i] & mask) == prefix:
                    local[(ordered[i] >> shift) & (radix - 1)] += 1
            with shared['hist_lock']:           # the only cross-CTA traffic:
                for b in range(radix):          # 256 counters, once per round
                    shared['hist'][cur][b] += local[b]
                if c == 0:
                    # Clear the NEXT round's buffer now, while everyone is
                    # still adding into this one.  That way the single barrier
                    # below proves two things at once - all adds for round r
                    # have landed, AND round r+1 starts from zero.
                    shared['hist'][nxt] = [0] * radix
            if not barrier.arrive_and_wait(phase, deadline):
                shared['deadlocked'] = True
                return
            phase += 1
            # Why three buffers and not two: a CTA that is slow to leave the
            # barrier is still reading round r's counts below, while the
            # others may already be adding into r+1 and clearing r+2.  With
            # only two buffers, "clear r+1" and "straggler still reading r-1"
            # are the same memory.  The third buffer is the slack.
            #
            # Every CTA now reads the SAME totals and does the SAME arithmetic,
            # so they all reach the same bucket without anyone broadcasting it.
            hist = list(shared['hist'][cur])
            ge = [0] * (radix + 1)
            for b in range(radix - 1, -1, -1):
                ge[b] = ge[b + 1] + hist[b]
            bucket = next(b for b in range(radix)
                          if ge[b] >= remaining and ge[b + 1] < remaining)
            prefix |= bucket << shift
            remaining -= ge[bucket + 1]
        pivot = prefix
        # -- collect.  Two passes, and the barrier between them is required. --
        gt = [i for i in mine if ordered[i] > pivot]
        with shared['out_lock']:                # reserve a contiguous run
            shared['out'].extend(gt)
        if not barrier.arrive_and_wait(phase, deadline):
            shared['deadlocked'] = True
            return
        phase += 1
        # Without that barrier a fast CTA's ties would eat output slots that a
        # slow CTA's guaranteed winners still need.
        eq = [i for i in mine if ordered[i] == pivot]
        with shared['out_lock']:
            room = k - len(shared['out'])
            shared['out'].extend(eq[:max(0, room)])
        per_cta[c] = {'chunk': (lo, hi), 'gt': len(gt), 'eq': len(eq)}
        # -- the exit protocol: who is allowed to reset the counter? ---------
        if reset_policy == 'first':
            # WRONG.  This CTA finished, but a peer may still be spinning
            # inside its final wait.  Zeroing the counter now means that peer
            # reads 0, which is less than its target, and it spins forever.
            if c == 0:
                with barrier.lock:
                    barrier.counter = 0
                shared['reset_by'] = c
        else:
            with barrier.lock:
                barrier.counter += 1
                mine_now = barrier.counter
            exit_target = (phase + 1) * num_ctas
            if mine_now == exit_target:         # exactly one CTA sees this,
                with barrier.lock:              # and it is provably the last
                    barrier.counter = 0
                shared['reset_by'] = c

    def hook_for(c):
        if slow_cta is None or c != slow_cta:
            return None

        def hook(spins):
            if spins == 0:
                time.sleep(0.05)                # deschedule at the worst moment
        return hook

    threads = []
    for c in range(num_ctas):
        h = hook_for(c)
        if h is None:
            t = threading.Thread(target=cta, args=(c,))
        else:
            def slow(cc=c, hh=h):
                orig = barrier.arrive_and_wait

                def patched(phase, dl, spin_hook=None):
                    return orig(phase, dl, hh)
                barrier.arrive_and_wait = patched
                try:
                    cta(cc)
                finally:
                    barrier.arrive_and_wait = orig
            t = threading.Thread(target=slow)
        threads.append(t)
    for t in threads:
        t.start()
    for t in threads:
        t.join(timeout + 1.0)

    info = {'num_ctas': num_ctas, 'chunk': chunk, 'per_cta': per_cta,
            'deadlocked': shared['deadlocked'], 'reset_by': shared['reset_by'],
            'counter_after': barrier.counter}
    if shared['deadlocked']:
        return None, info
    return shared['out'][:k], info

In [14]:
print('  num_ctas   chunk   correct   reset by   counter after')
for nc in (1, 2, 4, 8, 16):
    ok = 0
    for _ in range(3):
        got, info = multi_cta_topk(wide, 64, num_ctas=nc)
        if got is not None and sorted((wide[i] for i in got), reverse=True) == \
                sorted(wide, reverse=True)[:64]:
            ok += 1
    print(f'  {nc:>8}   {info["chunk"]:>5}     {ok}/3     {"CTA" + str(info["reset_by"]):>8}'
          f'   {info["counter_after"]:>13}')
    assert ok == 3

  num_ctas   chunk   correct   reset by   counter after
         1   20000     3/3         CTA0               0
         2   10000     3/3         CTA0               0
         4    5000     3/3         CTA0               0
         8    2500     3/3         CTA2               0
        16    1250     3/3        CTA15               0


The three details:

- The counter is **never reset between rounds**. Each CTA keeps a private phase
  and waits for `counter >= (phase + 1) * num_ctas`.
- The wait is `>=`, not `==`. A CTA that gets descheduled can wake up after the
  counter has already run past its target.
- The histogram is **triple** buffered. Round `r` accumulates into `hist[r % 3]`
  while CTA 0 clears `hist[(r + 1) % 3]`, so a single barrier proves both "every
  add landed" and "the next buffer is clean". The third buffer is slack for a
  straggler still reading round `r`. With one buffer this deadlocks; with two it
  races.

## 6. The failure case - flashinfer issue #3610

At the end of the kernel someone has to zero the counter so the next launch
starts clean. The obvious choice - whoever finishes first - is a deadlock: a
peer may still be spinning inside its final wait, and it reads 0, which is less
than its target, forever.

Below, CTA 3 is deliberately made slow so the hazard is deterministic rather
than a one-in-a-thousand flake - which is exactly what makes this class of bug
so unpleasant in the wild.

In [15]:
print('  reset policy   deadlocked   reset by')
for pol in ('last', 'first'):
    for _ in range(2):
        got, info = multi_cta_topk(wide, 64, num_ctas=4, reset_policy=pol,
                                   slow_cta=3, timeout=1.5)
        who = f'CTA{info["reset_by"]}' if info['reset_by'] is not None else '-'
        print(f'  {pol:<12}   {str(info["deadlocked"]):>10}   {who:>8}')
        assert info['deadlocked'] == (pol == 'first')

  reset policy   deadlocked   reset by
  last                False       CTA3
  last                False       CTA3
  first                True       CTA0
  first                True       CTA0


The fix is an exit barrier that doubles as leader election: every CTA
increments once more, and the single CTA whose increment lands exactly on the
target is provably the last one out - so every peer has already stopped
reading. Above, that is always CTA 3, the one we slowed down.

## 7. Ties, and two knobs people keep confusing

Radix select does not find "the k-th element". It finds the k-th **value**. If
the pivot occurs twelve times and five slots are left, seven of those elements
must lose, and nothing in the algorithm says which. On a GPU the answer is
decided by whichever block wins a race on an atomic, so the same input can
return a different - equally correct - answer on every run.

Two knobs fix two different halves of that, and they are not interchangeable:

- `deterministic` fixes the **order** the selected indices come out in.
- `tie_break` fixes **which** tied elements get selected at all.

In [16]:
def tie_topk(values, k, num_ctas=4, policy='race', rng=None):
    """Select k, and resolve ties at the pivot according to `policy`.

    'race'    whichever CTA gets to the atomic first  (what you get by default)
    'det'     fixed by CTA index, then position within the chunk
    'small'   prefer the smallest original index      (tie_break = Small)
    'large'   prefer the largest original index       (tie_break = Large)
    """
    rng = rng or random
    n = len(values)
    pivot, above, equal = radix_select(values, k)
    need = k - len(above)
    if need <= 0:
        return sorted(above[:k])
    # a grid-stride loop: CTA c owns elements c, c + num_ctas, c + 2*num_ctas, ...
    # This is how real kernels walk a row, and it is why the launch shape leaks
    # into the answer.
    lanes = [[i for i in equal if i % num_ctas == c] for c in range(num_ctas)]
    if policy == 'race':
        # CTAs reach the atomic in an unpredictable order; each dumps its ties
        order = list(range(num_ctas))
        rng.shuffle(order)
        pool = [i for c in order for i in lanes[c]]
    elif policy == 'det':
        # reproducible for a fixed launch - but the LANE ASSIGNMENT is baked
        # into the answer, so change num_ctas and the membership changes too
        pool = [i for c in range(num_ctas) for i in lanes[c]]
    elif policy == 'small':
        pool = sorted(equal)                     # row-global rule
    elif policy == 'large':
        pool = sorted(equal, reverse=True)
    else:
        raise ValueError(policy)
    return sorted(above + pool[:need])

In [17]:
tied = [f32(v) for v in ([9.0] * 3 + [5.0] * 12 + [1.0] * 10)]
kt = 8
rt = random.Random(3)

seen = {tuple(tie_topk(tied, kt, policy='race', rng=rt)) for _ in range(8)}
print(f'race            {len(seen)} different answers over 8 runs, all correct')

dets = {}
for nc in (2, 4, 8):
    s = {tuple(tie_topk(tied, kt, num_ctas=nc, policy='det')) for _ in range(6)}
    assert len(s) == 1
    dets[nc] = sorted(s)[0]
    print(f'det, {nc} CTAs      stable -> {dets[nc]}')
assert len(set(dets.values())) == 3

for pol in ('small', 'large'):
    s = {tuple(tie_topk(tied, kt, num_ctas=nc, policy=pol))
         for nc in (2, 4, 8) for _ in range(6)}
    assert len(s) == 1
    print(f'tie_break={pol:<6}  same answer for every launch shape -> {sorted(s)[0]}')

race            6 different answers over 8 runs, all correct
det, 2 CTAs      stable -> (0, 1, 2, 4, 6, 8, 10, 12)
det, 4 CTAs      stable -> (0, 1, 2, 4, 5, 8, 9, 12)
det, 8 CTAs      stable -> (0, 1, 2, 3, 8, 9, 10, 11)
tie_break=small   same answer for every launch shape -> (0, 1, 2, 3, 4, 5, 6, 7)
tie_break=large   same answer for every launch shape -> (0, 1, 2, 10, 11, 12, 13, 14)


Look at the middle block: `deterministic` gives a perfectly stable answer for
each launch shape, and **three different stable answers** across three shapes.
A grid-stride loop bakes `num_ctas` into which CTA sees which tie, so pinning
the output ordering does not pin membership. Only a row-global rule - smallest
original index wins, or largest - does, and that is why it costs more.

## 8. Which kernel does a library actually launch?

There is no single best top-k, so a library ships several and picks at launch
time from the shape of the problem.

In [18]:
def should_use_filtered(n, k, dtype='float32', smem_bytes=16 * 1024,
                        need_tie_break=False, cuda_graph=False,
                        num_rows=1, num_sms=132):
    """A stand-in for the dispatch logic in front of a real top-k.

    Returns (use_filtered, reason).  The filtered kernel is the fast path: one
    block owns a whole row, keeps the candidates in shared memory, and never
    talks to another block - no grid barrier, no second launch, no workspace.
    All the conditions below are about whether that is *possible*, not whether
    it is quick."""
    elem = 4 if dtype == 'float32' else 2
    capacity = smem_bytes // elem
    if k > capacity:
        return False, f'k={k} exceeds the {capacity}-element shared-memory budget'
    if k > 2048:
        return False, f'k={k} is past the point where a per-block sort stops paying'
    if need_tie_break:
        return False, 'tie_break needs a row-global rule, which needs the multi-CTA pass'
    if n > 64 * capacity:
        return False, f'n={n} makes the filter pass too likely to overflow'
    # One block per row means the grid is num_rows blocks wide.  A single
    # enormous row is the worst case: correct, fits, and uses 1 of 132 SMs.
    if num_rows * 4 < num_sms and n > 32_768:
        return False, (f'{num_rows} row(s) x {n} elements would leave '
                       f'{num_sms - num_rows} of {num_sms} SMs idle')
    # A CUDA-graph capture cannot branch on a device-side count, so a kernel
    # that might need "one more round" has to be sized for the worst case.
    if cuda_graph:
        return True, 'fits, and the round count is bounded so the graph is safe'
    return True, 'fits in shared memory, single block, no grid barrier'

In [19]:
cases = [(4096, 8, 'float32', False, 512), (200_000, 64, 'float32', False, 1),
         (4096, 8192, 'float32', False, 512), (4096, 64, 'float32', True, 512),
         (4096, 3000, 'float16', False, 512)]
for n_, k_, dt, tb, rows in cases:
    use, why = should_use_filtered(n_, k_, dt, need_tie_break=tb, num_rows=rows)
    print(f'n={n_:>7,} k={k_:>5} {dt:<8} rows={rows:<4} tie_break={str(tb):<5}')
    print(f'    -> {"filtered" if use else "multi-CTA":<10} : {why}')

n=  4,096 k=    8 float32  rows=512  tie_break=False
    -> filtered   : fits in shared memory, single block, no grid barrier
n=200,000 k=   64 float32  rows=1    tie_break=False
    -> multi-CTA  : 1 row(s) x 200000 elements would leave 131 of 132 SMs idle
n=  4,096 k= 8192 float32  rows=512  tie_break=False
    -> multi-CTA  : k=8192 exceeds the 4096-element shared-memory budget
n=  4,096 k=   64 float32  rows=512  tie_break=True 
    -> multi-CTA  : tie_break needs a row-global rule, which needs the multi-CTA pass
n=  4,096 k= 3000 float16  rows=512  tie_break=False
    -> multi-CTA  : k=3000 is past the point where a per-block sort stops paying


Most of those conditions are about whether the single-block path is *possible*
- shared memory, tie_break - and when it is, it wins by default: no grid
barrier, no second launch, no workspace. The odd one out is the second row,
which fits perfectly well; it is just that one block per row means one block,
and 131 idle SMs.

## 9. LeetCode 215 - Kth Largest Element in an Array

Quickselect, random pivot, three-way partition. The interview follow-up is the
part people miss.

In [20]:
def find_kth_largest(nums, k):
    """LC 215.  Quickselect with a random pivot and a three-way partition.

    The two traps, both of which turn O(n) into O(n^2):

      * a fixed pivot (first or last element) on already-sorted input
      * a two-way partition on input where every value is the same

    The second one is the nastier trap because sorted input at least looks
    suspicious in a test case, whereas [1,1,1,...,1] looks harmless."""
    a = quickselect(nums, k, rng=random.Random(0xC0FFEE))
    return a[k - 1]

In [21]:
print(find_kth_largest([3, 2, 1, 5, 6, 4], 2))
print(find_kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4))
assert find_kth_largest([3, 2, 1, 5, 6, 4], 2) == 5
assert find_kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4

t0 = time.time()
assert find_kth_largest([7] * 50_000, 25_000) == 7
print(f'50,000 identical values, k = 25,000 -> 7, in {time.time() - t0:.3f}s')

5
4
50,000 identical values, k = 25,000 -> 7, in 0.006s


That last case is the follow-up. A two-way partition splits `[7,7,7,...]` into
one empty side and one side of `n-1` - `O(n^2)`, roughly a billion operations
here. The three-way split drops every tie in a single pass.

It is the same fact that forced the GPU kernels above to think about ties at
all: a row of equal scores is not a pathological input you can dismiss, it is
what a masked attention row looks like.

## Complexity

| Approach | Time | Memory | Notes |
| --- | --- | --- | --- |
| sort, take k | O(n log n) | O(n) | answers a bigger question than asked |
| size-k min-heap | O(n log k) | O(k) | one pass, works on a stream |
| quickselect | O(n) average | O(1) extra | O(n^2) worst case without a random pivot |
| median-of-medians | O(n) worst case | O(n) | large constant, rarely shipped |
| radix select | O(n * rounds) | O(radix) | no comparisons, no data movement |
| filtered (1 block/row) | O(n) | O(candidates) | one global pass, then all on-chip |
| multi-CTA | O(n / ctas * rounds) | O(radix) | needs a grid-wide barrier |

Tomorrow the same bit-slicing idea gets pushed all the way: instead of
recursing into one bucket to *select*, keep every bucket and you have sorted
the whole array without a single comparison - and the parallel version of that
turns out to be a prefix sum.